<a href="https://www.kaggle.com/code/mylastresort/myspotify-top-100-by-genre?scriptVersionId=323448178" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 02 — Top 100 Most-Played Songs for a Given Genre

Returns the **100 globally most-played songs for a given majority genre**, sorted by total play count descending.

| Variable | Shape | Description |
|----------|-------|-------------|
| `tracks` | (1 000 000, 4) | track_id · song_id · artist · title |
| `genres` | (280 831, 3) | track_id · majority_genre · minority_genre |
| `triplets` | (48 373 586, 3) | user_id · song_id · play_count |
| `lyrics_long` | (16 845 822, 3) | track_id · word · count |

### Algorithm
1. Aggregate `play_count` (sum) per `song_id` from the triplets.
2. Join with `tracks` (via `song_id`) to obtain `track_id`, artist, and title.
3. Join with `genres` (via `track_id`) using only `majority_genre`.
4. Filter to the requested genre, keep the top 100 by total play count.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1–100 index, by total play count (descending) |
| `artist` | Artist name |
| `title` | Track title |
| `play_count` | Total plays across all users |


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class MySpotifyRecommender:
    tracks: pd.DataFrame
    genres: pd.DataFrame
    triplets: pd.DataFrame
    lyrics_long: pd.DataFrame

    # ── raw-file parsers ──────────────────────────────────────────────────

    @staticmethod
    def _parse_tracks(path: Path) -> pd.DataFrame:
        return pd.read_csv(
            path,
            sep="<SEP>",
            header=None,
            names=["track_id", "song_id", "artist", "title"],
            engine="python",
        )

    @staticmethod
    def _parse_genres(path: Path) -> pd.DataFrame:
        return pd.read_csv(
            path,
            sep="\t",
            comment="#",
            header=None,
            names=["track_id", "genre_major", "genre_minor"],
        )

    @staticmethod
    def _parse_triplets(path: Path) -> pd.DataFrame:
        df = pd.read_csv(
            path,
            sep="\t",
            header=None,
            names=["user_id", "song_id", "play_count"],
        )
        df["play_count"] = df["play_count"].astype(int)
        return df

    @staticmethod
    def _parse_lyrics(path: Path) -> pd.DataFrame:
        """Parse MXM sparse bag-of-words into long/tidy format (track_id, word, count)."""
        vocab_raw: list[str] = []
        data_lines: list[str] = []
        in_vocab = False
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                line = line.rstrip("\n")
                if line.startswith("#"):
                    continue
                if line.startswith("%") or in_vocab:
                    chunk = line.lstrip("%")
                    vocab_raw.append(chunk)
                    in_vocab = True
                    if ":" in line and not line.startswith("%"):
                        data_lines.append(line)
                        in_vocab = False
                        vocab_raw.pop()
                else:
                    if in_vocab:
                        in_vocab = False
                    data_lines.append(line)

        vocabulary = ",".join(vocab_raw).split(",")
        if vocabulary and vocabulary[0] in ("%i", "i", ""):
            vocabulary = vocabulary[1:]

        rows: list[dict] = []
        for line in data_lines:
            if not line.strip():
                continue
            parts = line.split(",")
            track_id = parts[0]
            for pair in parts[1:]:
                if ":" not in pair:
                    continue
                idx_str, cnt_str = pair.split(":", 1)
                idx = int(idx_str) - 1
                if 0 <= idx < len(vocabulary):
                    rows.append(
                        {
                            "track_id": track_id,
                            "word": vocabulary[idx],
                            "count": int(cnt_str),
                        }
                    )
        return pd.DataFrame(rows, columns=["track_id", "word", "count"])

    @staticmethod
    def _find_raw_file(root: Path, *names: str) -> Path | None:
        """Search root and common subdirs for a raw dataset file."""
        candidates = [root, root / "data"]
        if (root / "data").exists():
            candidates += [p for p in (root / "data").iterdir() if p.is_dir()]
        for base in candidates:
            if not base.exists():
                continue
            for name in names:
                p = base / name
                if p.exists():
                    return p
            for name in names:
                hits = list(base.glob(f"**/{name}"))
                if hits:
                    return hits[0]
        return None

    @classmethod
    def _build_csvs(cls, out_dir: Path, root: Path) -> None:
        """Parse raw MSD files and write the 4 CSVs to out_dir."""
        out_dir.mkdir(parents=True, exist_ok=True)
        tracks_raw = cls._find_raw_file(root, "p02_unique_tracks.txt")
        genres_raw = cls._find_raw_file(root, "p02_msd_tagtraum_cd2.cls")
        triplets_raw = cls._find_raw_file(root, "train_triplets.txt")
        lyrics_raw = cls._find_raw_file(root, "mxm_dataset_train.txt")

        missing_raw = [
            n
            for n, p in [
                ("tracks", tracks_raw),
                ("genres", genres_raw),
                ("triplets", triplets_raw),
                ("lyrics", lyrics_raw),
            ]
            if p is None
        ]
        if missing_raw:
            raise FileNotFoundError(f"Raw source files not found: {missing_raw}")

        steps = [
            ("tracks.csv", "Tracks", lambda: cls._parse_tracks(tracks_raw)),
            ("genres.csv", "Genres", lambda: cls._parse_genres(genres_raw)),
            ("triplets.csv", "Triplets", lambda: cls._parse_triplets(triplets_raw)),
            ("lyrics.csv", "Lyrics", lambda: cls._parse_lyrics(lyrics_raw)),
        ]
        for filename, label, parser in steps:
            dest = out_dir / filename
            print(f"  [{label}] parsing...", end=" ", flush=True)
            df = parser()
            df.to_csv(dest, index=False)
            print(f"{df.shape}  →  {dest.name}  ({dest.stat().st_size / 1e6:.0f} MB)")

    # ── constructor ───────────────────────────────────────────────────────

    @classmethod
    def from_files(
        cls, triplets_sample_rows: int | None = None
    ) -> MySpotifyRecommender:
        is_kaggle = Path("/kaggle/input").exists()
        cwd = Path.cwd()
        root = cwd.parent if (cwd.name == "notebooks" and not is_kaggle) else cwd

        _needed = {"tracks.csv", "genres.csv", "triplets.csv", "lyrics.csv"}

        def _find_csv_dir() -> Path | None:
            if is_kaggle:
                hits = list(Path("/kaggle/input").rglob("tracks.csv"))
                if hits:
                    candidate = hits[0].parent
                    if all((candidate / f).exists() for f in _needed):
                        return candidate
                return None
            return root / "data" / "csv"

        csv_dir = _find_csv_dir()
        if csv_dir is None or not all((csv_dir / f).exists() for f in _needed):
            if is_kaggle:
                raise FileNotFoundError(
                    "CSVs not found under /kaggle/input. "
                    "Add 'mylastresort/p02-myspotify' as a dataset source."
                )
            csv_dir = root / "data" / "csv"
            missing = [f for f in _needed if not (csv_dir / f).exists()]
            if missing:
                print(f"Missing CSVs {missing} — building from raw files ...")
                cls._build_csvs(csv_dir, root)

        print(f"Environment : {'Kaggle' if is_kaggle else 'Local'}")
        print(f"CSV dir     : {csv_dir}")

        tracks = pd.read_csv(csv_dir / "tracks.csv")
        genres = pd.read_csv(csv_dir / "genres.csv").rename(
            columns={"genre_major": "majority_genre", "genre_minor": "minority_genre"}
        )
        triplets = pd.read_csv(
            csv_dir / "triplets.csv",
            dtype={"play_count": "int32"},
            nrows=triplets_sample_rows,
        )
        lyrics_long = pd.read_csv(csv_dir / "lyrics.csv")

        print(f"\n  tracks      {tracks.shape}")
        print(f"  genres      {genres.shape}")
        print(f"  triplets    {triplets.shape}")
        print(f"  lyrics_long {lyrics_long.shape}")

        return cls(
            tracks=tracks, genres=genres, triplets=triplets, lyrics_long=lyrics_long
        )

    # ── Add your research methods here ─────────────────────────────────────

    def top_per_genre(self, genre: str, n: int = 100) -> pd.DataFrame:
        """Return the top n most-played songs for the given majority genre.

        Parameters
        ----------
        genre : str
            The majority genre label to filter by (case-sensitive).
        n : int
            Maximum number of songs to return (default 100).

        Returns a DataFrame with columns:
            artist, title, play_count
        and a 1-based integer index named 'rank', sorted by play_count descending.
        """
        # 1. Aggregate play counts per song
        song_plays = self.triplets.groupby("song_id", as_index=False).agg(
            play_count=("play_count", "sum")
        )

        # 2. Join with tracks to get track_id, artist, title
        tracks_dedup = self.tracks.drop_duplicates(subset="song_id")[
            ["song_id", "track_id", "artist", "title"]
        ]
        enriched = song_plays.merge(tracks_dedup, on="song_id", how="inner")

        # 3. Filter to the requested majority genre only
        genre_map = self.genres[["track_id", "majority_genre"]].dropna(
            subset=["majority_genre"]
        )
        enriched = enriched.merge(genre_map, on="track_id", how="inner")
        enriched = enriched[enriched["majority_genre"] == genre]

        # 4. Sort by play count descending and keep top n
        result = (
            enriched.sort_values("play_count", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        result.index += 1
        result.index.name = "rank"

        return result[["artist", "title", "play_count"]]

In [2]:
rs = MySpotifyRecommender.from_files()

Environment : Kaggle
CSV dir     : /kaggle/input/datasets/mylastresort/p02-myspotify

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845822, 3)


---
## Research

In [3]:
# list all genres
rs.genres["majority_genre"].dropna().unique()

array(['Rock', 'Rap', 'Latin', 'Jazz', 'Electronic', 'Punk', 'Pop',
       'New Age', 'Metal', 'RnB', 'Country', 'Reggae', 'Folk', 'Blues',
       'World'], dtype=object)

In [4]:
top100_rock = rs.top_per_genre("Rock", 100)
top100_rock


,artist,title,play_count
rank,,,
1,Björk,Undo,648239
2,Kings Of Leon,Revelry,527893
3,Harmonia,Sehr kosmisch,425463
4,OneRepublic,Secrets,292642
5,Tub Ring,Invalid,268353
...,...,...,...
96,Metric,Gold Guns Girls,28148
97,Pearl Jam,Encore Break,27579
98,Daughtry,No Surprise,27187
